# Segment scaling: MESSI, SOFA, and SPARTAN

This notebook reads the archives produced by `run_segment_scaling_experiment.sh`. For every dataset and segment count, it selects the faster depth/width result for SOFA and SPARTAN, then reports one macro-averaged query time for each system at 16, 32, and 64 symbolic dimensions. For SPARTAN, the x-axis uses the record-bound prefix dimensions rather than the fixed 128-dimensional MBR width.

In [ ]:
from pathlib import Path
import re
import pandas as pd
import matplotlib.pyplot as plt

ROOT_CANDIDATES = [
    Path('trie_logs/segment_scaling'),
    Path('notebooks/trie_logs/segment_scaling'),
    Path('results/segment_scaling'),
]
LOG_ROOT = next((path for path in ROOT_CANDIDATES if path.exists()), None)
if LOG_ROOT is None:
    raise FileNotFoundError(
        'Could not find trie_logs/segment_scaling or results/segment_scaling. '
        'Run from the repository root or the notebooks directory.'
    )

def _settings_map(path):
    values = {}
    for line in path.read_text(errors='replace').splitlines()[1:]:
        if ',' not in line:
            continue
        key, value = line.split(',', 1)
        values[key.strip().strip(chr(34)).lower()] = value.strip().strip(chr(34))
    return values

def _settings_file(query_file):
    stamp = query_file.stem.replace('MESSI_QUERY_', '')
    candidate = query_file.parent.parent / 'settings' / f'MESSI_SETTINGS_{stamp}.csv'
    if candidate.exists():
        return candidate
    matches = list(LOG_ROOT.rglob(f'MESSI_SETTINGS_{stamp}.csv'))
    return matches[0] if matches else None

def _system(query_file):
    return next((part for part in query_file.parts if part in {'messi', 'sofa', 'trie'}), 'unknown')

def _segments(query_file, system, settings):
    key = 'record lower-bound dimensions' if system == 'trie' else 'symbolic dimensions'
    setting_value = settings.get(key)
    setting_segments = int(setting_value) if setting_value and setting_value.isdigit() else None
    path_match = next((re.fullmatch(r'segments-(16|32|64)', part) for part in query_file.parts
                       if re.fullmatch(r'segments-(16|32|64)', part)), None)
    path_segments = int(path_match.group(1)) if path_match else None
    if setting_segments is not None and path_segments is not None and setting_segments != path_segments:
        raise ValueError(f'Segment mismatch for {query_file}: settings={setting_segments}, path={path_segments}')
    return setting_segments if setting_segments is not None else path_segments

def _method(system, settings):
    histogram = settings.get('histogram type', '')
    variant = {'1': 'depth', '2': 'width'}.get(histogram, 'single')
    if system == 'messi':
        return 'MESSI / SAX'
    if system == 'sofa':
        return f'SOFA / SFA-{variant}'
    if system == 'trie':
        return f'TRIE / SPARTAN-{variant}'
    return f'unknown-{variant}'

def load_segment_scaling(root=LOG_ROOT):
    rows = []
    for query_file in sorted(root.rglob('MESSI_QUERY_*.csv')):
        settings_file = _settings_file(query_file)
        if settings_file is None:
            continue
        settings = _settings_map(settings_file)
        system = _system(query_file)
        segments = _segments(query_file, system, settings)
        if system == 'unknown' or segments not in (16, 32, 64):
            continue
        frame = pd.read_csv(query_file)
        if len(frame) > 1:
            frame = frame.iloc[:-1].copy()  # MESSI's final row is an aggregate
        if 'querying time' not in frame.columns:
            continue
        latency_ms = pd.to_numeric(frame['querying time'], errors='coerce').dropna() / 1000.0
        if latency_ms.empty:
            continue
        dataset_value = settings.get('dataset', query_file.parents[2].name)
        rows.append({
            'system': system,
            'method_variant': _method(system, settings),
            'dataset': Path(dataset_value).stem or query_file.parents[2].name,
            'segments': segments,
            'query_ms': float(latency_ms.mean()),
            'queries': int(latency_ms.size),
            'query_threads': int(settings.get('threads', 0) or 0),
            'index_threads': int(settings.get('index threads', 0) or 0),
            'simd_enabled': settings.get('simd', '0') not in {'0', 'false', 'False', ''},
            'source': str(query_file),
        })
    result = pd.DataFrame(rows)
    if result.empty:
        raise RuntimeError(f'No segment-scaling query files found below {root}')
    return result

runs = load_segment_scaling()

# Average duplicate archives of the same variant first. Then choose only
# between depth and width for each dataset/segment pair.
variants = (runs.groupby(['system', 'method_variant', 'dataset', 'segments'], as_index=False)
            .agg(query_ms=('query_ms', 'mean'),
                 files=('source', 'count'),
                 query_threads=('query_threads', 'median'),
                 index_threads=('index_threads', 'median'),
                 simd_enabled=('simd_enabled', 'all')))
best = variants.loc[variants.groupby(['system', 'dataset', 'segments'])['query_ms'].idxmin()].copy()
best['method'] = best['system'].map({
    'messi': 'MESSI / SAX',
    'sofa': 'SOFA / SFA',
    'trie': 'TRIE / SPARTAN',
})

summary = (best.groupby(['method', 'segments'], as_index=False)
           .agg(datasets=('dataset', 'nunique'),
                mean_query_ms=('query_ms', 'mean'),
                sd_query_ms=('query_ms', 'std'),
                query_threads=('query_threads', 'median'),
                index_threads=('index_threads', 'median'),
                simd_all_runs=('simd_enabled', 'all')))
summary['sd_query_ms'] = summary['sd_query_ms'].fillna(0.0)
summary = summary.sort_values(['method', 'segments'])

pd.set_option('display.float_format', lambda value: f'{value:,.3f}')
print(f'Loaded {len(runs)} query files from {LOG_ROOT}')
print('Depth/width is selected per dataset before the macro-average. Query time is in ms.')
display(summary)

wide = summary.pivot(index='method', columns='segments', values='mean_query_ms').reindex(columns=[16, 32, 64])
print('\nCompact table (mean query ms):')
display(wide)

fig, ax = plt.subplots(figsize=(8.5, 5.0))
for method, group in summary.groupby('method', sort=True):
    group = group.sort_values('segments')
    ax.plot(group['segments'], group['mean_query_ms'], marker='o', linewidth=2, label=method)
ax.set_xticks([16, 32, 64])
ax.set_yscale('log')
ax.set_xlabel('Symbolic dimensions / n_segments')
ax.set_ylabel('Mean query time (ms, log scale)')
ax.set_title('Query time versus symbolic dimensionality')
ax.grid(axis='both', alpha=0.3)
ax.legend(frameon=False)
fig.tight_layout()
plt.show()

# Optional machine-readable output next to the copied logs.
# summary.to_csv(LOG_ROOT / 'segment_scaling_summary.csv', index=False)